In [1]:
import dotenv
import os, io, hashlib
from huggingface_hub import hf_hub_download
from pathlib import Path
from openai import OpenAI

In [2]:
# In der RENKU-Umgebung muss man nicht diese Zeile laufen lassen
dotenv.load_dotenv()

True

# Download Audio-File from the Huggingface-Repo

In [3]:
REPO_ID = "nkamiy/workshop2025-media"

SHA_WAV_RAMROD = os.getenv("SHA_WAV_RAMROD")

FILE = ["ramrod_ausschnitt_kurz.wav", f"sha256:{SHA_WAV_RAMROD}"]


def sha256sum(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


local = hf_hub_download(
        repo_id=REPO_ID,
        filename=FILE[0],
        repo_type="dataset",   
        local_dir="../data"
    )

print(f"downloaded: {local}")

if FILE[1].startswith("sha256:"):
    got = sha256sum(Path(local))
    assert got == FILE[1].split(":", 1)[1], f"Checksum mismatch for {FILE[0]}"
print("All files ready.")

downloaded: ../data/ramrod_ausschnitt_kurz.wav
All files ready.


# Zuerst ohne Diarization

In [4]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [5]:
with open(local, "rb") as audio_file:
    transcript = client.audio.transcriptions.create(
        model="gpt-4o-transcribe",
        file=audio_file,
        chunking_strategy="auto",
    )

In [6]:
print(transcript.text)

Hello, Bill.
Hello Connie.
Where is everybody?
round
可以。
Sure could do with a cup of coffee.
Did Burma really draw first?
George said so, didn't he?
Dere.
Well, what do you think?
I think you're not afraid to take a chance.
I like this.
What are you after, Connie?
All right, I'll tell you.
I want to break Frank Ivy.
I want to see him crawl out of town the same way Walt did.
If Curly dies
Dave will take care of Ivy.
Dave's way is too slow.
курва
Just what's on your mind, Connie?
I want you to stampede my herd.
Stampede?
The one way to make Dave and Jim crew move fast.
Thought about it for a long time.
They'll think Frank Ivey did it.
They might.
This is something I didn't figure on.
What did you think?
To tell the truth,
Why bother?


# Diarization (Neu seit Okt. 2025 )

In [7]:
with open(local, "rb") as audio_file:
    transcript_diarized = client.audio.transcriptions.create(
        model="gpt-4o-transcribe-diarize",
        file=audio_file,
        response_format="diarized_json",
        chunking_strategy="auto",
    )

Unexpected audio response format: diarized_json


In [8]:
# Trotz der Meldung "Unexpected audio response format: diarized_json" erhalten wir die Antwort
print(transcript_diarized)

Transcription(text="Hello, Bill. Hello, Connie. Where is everybody? Around. Sure could do with a cup of coffee. Bill, did Burma really draw first?\nGeorge said so, didn't he? Did he? Well, what do you think? I think you're not afraid to take a chance. I like that. What are you after, Connie? All right, I'll tell you. I want to break Frank Ivy. I want to see him crawl out of town the same way Walt did.\nIf Curly dies, Dave will take care of Ivy. Dave's way's too slow. Just what's on your mind, Connie? I want you to stampede my herd. It's the one way to make Dave and Jim Crew move fast. I've thought about it for a long time.\nThey'll think Frank Ivy did it. Yeah, they might. But they will. This was something I didn't figure out. What did you figure, Bill? Well, to tell the truth, I... Why bother?", logprobs=None, usage=UsageTokens(input_tokens=1303, output_tokens=2883, total_tokens=4186, type='tokens', input_token_details=UsageTokensInputTokenDetails(audio_tokens=1228, text_tokens=75)), 

In [9]:
for segment in transcript_diarized.segments:
    print(segment["speaker"], segment["text"], segment["start"], segment["end"])

A  Hello, Bill. 7.252 7.901999999999999
B  Hello, 9.002 9.501999999999999
B  Connie. 9.652000000000001 9.852
B  Where is everybody? 10.652000000000001 11.452
A  Around. 12.952 13.652000000000001
B  Sure could do with a cup of coffee. 15.252000000000002 16.652
A  Bill, 29.652 29.852
A  did Burma really draw first? 32.402 33.702000000000005
B  George said so, didn't he? 35.22 36.269999999999996
A  Did he? 37.12 37.67
B  Well, what do you think? 38.42 39.269999999999996
A  I think you're not afraid to take a chance. 43.72 45.62
A  I like that. 47.019999999999996 47.97
B  What are you after, Connie? 50.97 52.12
A  All right, I'll tell you. 54.519999999999996 55.519999999999996
A  I want to break Frank Ivy. 56.519999999999996 58.019999999999996
A  I want to see him crawl out of town the same way Walt did. 58.72 61.269999999999996
B  If Curly dies, 62.356 63.556000000000004
B  Dave will take care of Ivy. 64.456 65.706
A  Dave's way's too slow. 67.606 68.856
B  Just what's on your mind, Conni